In [ ]:
import numpy as np
import pandas as pd

def quick_benchmark_with_metrics(num_circuits=10):
    """
    Benchmark loss functions and track individual metric improvements
    
    Returns:
        results: Dict of improvements per loss function
        metrics_results: DataFrame with detailed metric changes
    """
    
    loss_functions = {
        'naive_loss': loss.naive_lossa,
        'informed_loss': loss.informed_lossa,
        'quadratic_loss_circuit': loss.quadratic_loss_circuit,
    }
    
    # Store loss improvements
    results = {name: [] for name in loss_functions.keys()}
    
    # Store detailed metrics
    metrics_data = []
    
    for i in range(num_circuits):
        print(f"Circuit {i+1}/{num_circuits}...", end=' ')
        
        # Generate circuit
        circ = random_clifford_t_circuit_controlled(50, 30, t_probability=0.5)
        diagram = integration.qiskit_to_pyzx(circ)

        # Get initial circuit metrics
        initial_circuit = integration.pyzx_to_qiskit(diagram)
        initial_metrics = {
            'qubits': metrics.circuit_qubit_count(initial_circuit),
            'total_gates': metrics.circuit_gate_count(initial_circuit),
            'two_qubit_gates': metrics.circuit_two_qubit_gate_count(initial_circuit),
            'clifford_gates': metrics.circuit_clifford_gate_count(initial_circuit),
            't_gates': metrics.circuit_t_gate_count(initial_circuit),
            'depth': initial_circuit.depth(),
        }
        
        # Test each loss function
        for loss_name, loss_func in loss_functions.items():
            wrapped_loss = loss.cost_function_from_circuit(
                loss_func,
                integration.pyzx_to_qiskit
            )
            
            initial_cost = wrapped_loss(diagram)
            
            best_diagram, best_cost, _ = simulated_annealing.simulated_annealing_zx(
                diagram=diagram,
                cost_function=wrapped_loss,
                get_neighbor=simulated_annealing.get_neighbor_weighted_rules,
                initial_temp=10.0,
                cooling_rate=0.99,
                max_iterations=500,
                max_no_improvement=50,
                verbose=False
            )
            
            # Get final circuit metrics
            final_circuit = integration.pyzx_to_qiskit(best_diagram)
            final_metrics = {
                'qubits': metrics.circuit_qubit_count(final_circuit),
                'total_gates': metrics.circuit_gate_count(final_circuit),
                'two_qubit_gates': metrics.circuit_two_qubit_gate_count(final_circuit),
                'clifford_gates': metrics.circuit_clifford_gate_count(final_circuit),
                't_gates': metrics.circuit_t_gate_count(final_circuit),
                'depth': final_circuit.depth(),
            }
            
            # Calculate improvements
            improvement = initial_cost - best_cost
            results[loss_name].append(improvement)
            
            # Store detailed metrics
            metrics_data.append({
                'circuit_id': i,
                'loss_function': loss_name,
                'initial_cost': initial_cost,
                'final_cost': best_cost,
                'cost_improvement': improvement,
                # Initial metrics
                'initial_qubits': initial_metrics['qubits'],
                'initial_total_gates': initial_metrics['total_gates'],
                'initial_two_qubit_gates': initial_metrics['two_qubit_gates'],
                'initial_clifford_gates': initial_metrics['clifford_gates'],
                'initial_t_gates': initial_metrics['t_gates'],
                'initial_depth': initial_metrics['depth'],
                # Final metrics
                'final_qubits': final_metrics['qubits'],
                'final_total_gates': final_metrics['total_gates'],
                'final_two_qubit_gates': final_metrics['two_qubit_gates'],
                'final_clifford_gates': final_metrics['clifford_gates'],
                'final_t_gates': final_metrics['t_gates'],
                'final_depth': final_metrics['depth'],
                # Improvements (negative means reduction)
                'qubits_reduction': initial_metrics['qubits'] - final_metrics['qubits'],
                'total_gates_reduction': initial_metrics['total_gates'] - final_metrics['total_gates'],
                'two_qubit_gates_reduction': initial_metrics['two_qubit_gates'] - final_metrics['two_qubit_gates'],
                'clifford_gates_reduction': initial_metrics['clifford_gates'] - final_metrics['clifford_gates'],
                't_gates_reduction': initial_metrics['t_gates'] - final_metrics['t_gates'],
                'depth_reduction': initial_metrics['depth'] - final_metrics['depth'],
            })
        
        print("Done")
    
    # Convert to DataFrame
    df = pd.DataFrame(metrics_data)
    
    # Print cost improvement summary
    print("\n" + "="*70)
    print("AVERAGE COST IMPROVEMENT PER LOSS FUNCTION")
    print("="*70)
    for loss_name, improvements in results.items():
        avg = np.mean(improvements)
        std = np.std(improvements)
        print(f"{loss_name:20s}: {avg:6.3f} ± {std:6.3f}")
    
    # Print metric reduction summary
    print("\n" + "="*70)
    print("AVERAGE METRIC REDUCTIONS PER LOSS FUNCTION")
    print("="*70)
    
    # Only show the most important metrics
    key_metric_columns = [
        ('qubits_reduction', 'Qubit Count'),
        ('depth_reduction', 'Circuit Depth'),
        ('total_gates_reduction', 'Gate Count'),
        ('two_qubit_gates_reduction', 'Two-Qubit Gate Count'),
        ('t_gates_reduction', 'T Gate Count')
    ]
    
    for loss_name in loss_functions.keys():
        print(f"\n{loss_name}:")
        print("-" * 50)
        loss_df = df[df['loss_function'] == loss_name]
        
        for metric_col, metric_name in key_metric_columns:
            avg = loss_df[metric_col].mean()
            std = loss_df[metric_col].std()
            print(f"  {metric_name:25s}: {avg:6.2f} ± {std:6.2f}")
    
    # Print detailed comparison table with only key metrics
    print("\n" + "="*70)
    print("METRIC REDUCTION COMPARISON (AVERAGE ACROSS ALL CIRCUITS)")
    print("="*70)
    
    key_columns = [col for col, _ in key_metric_columns]
    summary = df.groupby('loss_function')[key_columns].mean().round(2)
    
    # Rename columns for display
    summary.columns = [name for _, name in key_metric_columns]
    print(summary.to_string())
    
    # Show which loss function is best for each metric
    print("\n" + "="*70)
    print("BEST LOSS FUNCTION PER METRIC")
    print("="*70)
    for metric_col, metric_name in key_metric_columns:
        best_loss = df.groupby('loss_function')[metric_col].mean().idxmax()
        best_value = df.groupby('loss_function')[metric_col].mean().max()
        print(f"{metric_name:30s}: {best_loss:20s} ({best_value:6.2f})")
    
    return results, df

# Run the benchmark
results, metrics_df = quick_benchmark_with_metrics(num_circuits=10)

# Optionally save to CSV
metrics_df.to_csv('detailed_metrics_benchmark.csv', index=False)
print("\n✓ Detailed results saved to 'detailed_metrics_benchmark.csv'")

In [ ]:
import numpy as np
def quick_benchmark(num_circuits=10):
    """Quick benchmark - just show averages"""
    
    loss_functions = {
        'naive_loss': loss.naive_loss,
        'informed_loss': loss.informed_loss,
        'quadratic_loss': loss.quadratic_loss_circuit,
    }
    
    # Store results
    results = {name: [] for name in loss_functions.keys()}
    
    for i in range(num_circuits):
        print(f"Circuit {i+1}/{num_circuits}...", end=' ')
        
        # Generate circuit
        circ = random_clifford_t_circuit_controlled(5, 20, t_probability=0.5)
        diagram = integration.qiskit_to_pyzx(circ)
        
        # Test each loss function
        for loss_name, loss_func in loss_functions.items():
            wrapped_loss = loss.cost_function_from_circuit(
                loss_func,
                integration.pyzx_to_qiskit
            )
            
            initial_cost = wrapped_loss(diagram)
            
            best_diagram, best_cost, _ = simulated_annealing.simulated_annealing_zx(
                diagram=diagram,
                cost_function=wrapped_loss,
                get_neighbor=simulated_annealing.get_neighbor_random_vertex_random_rule,
                initial_temp=100.0,
                cooling_rate=0.95,
                max_iterations=500,
                max_no_improvement=50,
                verbose=False
            )
            
            improvement = initial_cost - best_cost
            results[loss_name].append(improvement)
        
        print("Done")
    
    # Print averages
    print("\n" + "="*50)
    print("AVERAGE IMPROVEMENT PER LOSS FUNCTION")
    print("="*50)
    for loss_name, improvements in results.items():
        avg = np.mean(improvements)
        std = np.std(improvements)
        print(f"{loss_name:20s}: {avg:6.3f} ± {std:6.3f}")
    
    return results

# Run it
results = quick_benchmark(num_circuits=10)